# Exp 00 — EDA + Preprocessing датасета Slovo RSL

**Google Colab Pro** | CPU-задача (MediaPipe) | Время: ~6–8 ч

## Что делает этот ноутбук
1. **EDA**: распределение классов, длины жестов, дикторы → MLflow
2. **Preprocessing**: видео → MediaPipe (Hands + Pose) → numpy `(N, 64, 75, 3)` → Google Drive
3. **Артефакты**: `label_mapping.json`, `class_names.json`, `features.npy`, `labels.npy` → DVC push

## Исправлено по сравнению с предыдущим запуском на Kaggle
| Проблема | Было | Стало |
|----------|------|-------|
| Кол-во кадров | `target_frames=8` | `sequence_length=64` (params.yaml) |
| Val видео (0/5000) | искал только в `val/` — не существует | `find_video()` ищет в `train/`, `test/`, `val/`, root |
| Сохранение | только `/kaggle/working` (ephemeral) | Drive + DVC push |
| MediaPipe режим | `static_image_mode=True` (медленнее) | `static_image_mode=False` (3-5x быстрее) |
| Resumability | нет | checkpoint каждые 500 видео |
| Среда | Kaggle Secrets | Colab Secrets |

## Перед запуском — Colab Secrets (🔑)
| Secret | Значение |
|--------|----------|
| `DAGSHUB_TOKEN` | токен DAGsHub (Settings → Access Tokens) |
| `MLFLOW_TRACKING_USERNAME` | `noviyblock` |
| `MLFLOW_TRACKING_PASSWORD` | тот же токен DAGsHub |
| `KAGGLE_USERNAME` | логин Kaggle (для скачивания датасета) |
| `KAGGLE_KEY` | API-ключ Kaggle (Account → Create New API Token) |

In [ ]:
# ── 1. Зависимости ────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'dagshub', 'mlflow', 'pyyaml', 'kaggle',
    'protobuf>=4.25.3,<5', 'mediapipe==0.10.14',
    'opencv-python-headless', 'seaborn', 'scikit-learn', 'tqdm',
], check=False)
print('Done')

In [ ]:
# ── 2. Drive + Colab Secrets + DAGsHub ───────────────────────────────────────
import os
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = '/content/drive/MyDrive/glossa'
DATA_ROOT  = f'{DRIVE_ROOT}/data'
SLOVO_DIR  = f'{DATA_ROOT}/slovo'
OUT_DIR    = f'{DATA_ROOT}/gestures/processed'
CKPT_DIR   = f'{DRIVE_ROOT}/.preprocess_checkpoint'

for d in [SLOVO_DIR, f'{OUT_DIR}/train', f'{OUT_DIR}/val', CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

for key in ('DAGSHUB_TOKEN', 'MLFLOW_TRACKING_USERNAME', 'MLFLOW_TRACKING_PASSWORD'):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f'[warn] Secret {key} not found')

token = os.environ.get('DAGSHUB_TOKEN') or os.environ.get('MLFLOW_TRACKING_PASSWORD', '')
if token:
    os.environ.setdefault('AWS_ACCESS_KEY_ID', token)
    os.environ.setdefault('AWS_SECRET_ACCESS_KEY', token)
    os.environ.setdefault('MLFLOW_S3_ENDPOINT_URL', 'https://dagshub.com/noviyblock/glossa.s3')

try:
    import dagshub
    dagshub.init(repo_owner='noviyblock', repo_name='glossa', mlflow=True)
    print('[DAGsHub] OK')
except Exception as e:
    import mlflow
    mlflow.set_tracking_uri('https://dagshub.com/noviyblock/glossa.mlflow')
    print(f'[MLflow] fallback: {e}')

import mlflow
mlflow.set_experiment('00_glossa_eda_and_preprocessing')
print(f'Output dir: {OUT_DIR}')

In [ ]:
# ── 3. Скачивание датасета Slovo с Kaggle ────────────────────────────────────
from pathlib import Path
import subprocess

SLOVO_ROOT  = Path(SLOVO_DIR)
ANNOTATIONS = SLOVO_ROOT / 'annotations.csv'

if ANNOTATIONS.exists():
    print(f'Датасет уже есть: {SLOVO_ROOT}')
else:
    print('Скачиваем Slovo с Kaggle...')
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception:
        print('[warn] KAGGLE_USERNAME / KAGGLE_KEY не найдены в Secrets')
        print('Поместите датасет вручную: Drive/glossa/data/slovo/annotations.csv + train/')
        raise SystemExit('Нет датасета')

    r = subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'kapitanov/slovo',
        '-p', str(SLOVO_ROOT),
        '--unzip',
    ], capture_output=True, text=True)
    print('OK' if r.returncode == 0 else r.stderr[:300])

subdirs = [p.name for p in SLOVO_ROOT.iterdir() if p.is_dir()]
print(f'Структура датасета: {subdirs}')
print(f'annotations.csv: {ANNOTATIONS.exists()}')

In [ ]:
# ── 4. EDA → MLflow ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, time
from pathlib import Path

df = pd.read_csv(SLOVO_ROOT / 'annotations.csv', sep='\t', on_bad_lines='skip')
df = df[df['text'] != 'no_event'].reset_index(drop=True)  # убираем 20x-дисбаланс класс
df['seq_len'] = df['end'] - df['begin'] + 1
df['split']   = df['train'].apply(lambda x: 'train' if bool(x) else 'val')

print('=' * 55)
print('EDA — Slovo RSL')
print('=' * 55)
print(f'Всего записей:       {len(df)}')
print(f'Уникальных жестов:   {df["text"].nunique()}')
print(f'Уникальных дикторов: {df["user_id"].nunique()}')
print(f'Train:               {(df.split=="train").sum()}')
print(f'Val:                 {(df.split=="val").sum()}')
print(f'Средняя длина:       {df.seq_len.mean():.1f} кадров')
print(f'Медиана:             {df.seq_len.median():.0f} кадров')
print(f'P75:                 {df.seq_len.quantile(.75):.0f} кадров  ← basis for seq_len=64')
print(f'Дисбаланс:           {df.text.value_counts().iloc[0] / df.text.value_counts().iloc[-1]:.0f}x')

EDA_DIR = f'{OUT_DIR}/eda'
os.makedirs(EDA_DIR, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df.seq_len, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(64, color='red', ls='--', lw=2, label='target=64')
axes[0].axvline(df.seq_len.median(), color='orange', ls='--', lw=1.5,
                label=f'median={df.seq_len.median():.0f}')
axes[0].set_xlabel('Длина жеста (кадры)'); axes[0].set_ylabel('Кол-во')
axes[0].set_title('Распределение длин жестов'); axes[0].legend()
top20 = df.text.value_counts().head(20)
axes[1].barh(range(20), top20.values, color='steelblue')
axes[1].set_yticks(range(20)); axes[1].set_yticklabels(top20.index, fontsize=8)
axes[1].invert_yaxis(); axes[1].set_title('Топ-20 классов')
plt.tight_layout()
plt.savefig(f'{EDA_DIR}/eda_overview.png', dpi=120)
plt.show()

with mlflow.start_run(run_name=f'eda_{time.strftime("%Y%m%d_%H%M")}') as run:
    mlflow.log_metrics({
        'total_samples':   len(df),
        'num_classes':     df.text.nunique(),
        'num_speakers':    df.user_id.nunique(),
        'seq_len_mean':    float(df.seq_len.mean()),
        'seq_len_median':  float(df.seq_len.median()),
        'seq_len_p75':     float(df.seq_len.quantile(.75)),
        'class_imbalance': float(df.text.value_counts().iloc[0] / df.text.value_counts().iloc[-1]),
    })
    mlflow.log_artifact(f'{EDA_DIR}/eda_overview.png')
print(f'EDA: {run.info.run_id}')

In [ ]:
# ── 5. Label mapping ──────────────────────────────────────────────────────────
import json, time

unique_labels = sorted(df['text'].unique())
label_to_id   = {label: idx for idx, label in enumerate(unique_labels)}

MAPPING_PATH = f'{OUT_DIR}/label_mapping.json'
NAMES_PATH   = f'{OUT_DIR}/class_names.json'

with open(MAPPING_PATH, 'w', encoding='utf-8') as f:
    json.dump({'label_to_id': label_to_id,
               'id_to_label': {str(v): k for k, v in label_to_id.items()},
               'num_classes': len(unique_labels)}, f, ensure_ascii=False, indent=2)
with open(NAMES_PATH, 'w', encoding='utf-8') as f:
    json.dump(unique_labels, f, ensure_ascii=False, indent=2)

print(f'Классов: {len(unique_labels)}')
print(f'Примеры: {unique_labels[:8]}')

with mlflow.start_run(run_name=f'label_mapping_{time.strftime("%Y%m%d_%H%M")}') as run:
    mlflow.log_param('num_classes', len(unique_labels))
    mlflow.log_artifact(MAPPING_PATH)
    mlflow.log_artifact(NAMES_PATH)
print('label_mapping.json залогирован')

In [ ]:
# ── 6. MediaPipe — инициализация и helpers ────────────────────────────────────
import mediapipe as mp
import cv2
import numpy as np
from pathlib import Path

SEQ_LEN = 64  # params.yaml: gesture.sequence_length = 64


def find_video(attachment_id: str, slovo_root: Path) -> str | None:
    """Ищет видео в нескольких возможных папках.

    КЛЮЧЕВОЕ ИСПРАВЛЕНИЕ: в Slovo val-видео физически лежат в той же папке
    train/ (или test/), разделение задаётся только через CSV-колонку `train`.
    Оригинальный ноутбук искал только в val/ -> 0/5000 успешно.
    """
    fname = f'{attachment_id}.mp4'
    for subdir in ['train', 'test', 'val', '']:
        p = slovo_root / subdir / fname
        if p.exists():
            return str(p)
    return None


def extract_keypoints(frames: list) -> np.ndarray:
    """frames: list of (H,W,3) uint8 RGB -> (T, 75, 3) float16.

    static_image_mode=False (video mode) — 3-5x быстрее для последовательных кадров.
    Layout: 0-32 = Pose, 33-53 = Left Hand, 54-74 = Right Hand.
    """
    with mp.solutions.hands.Hands(
        static_image_mode=False, max_num_hands=2,
        model_complexity=1, min_detection_confidence=0.4,
        min_tracking_confidence=0.4,
    ) as hands, mp.solutions.pose.Pose(
        static_image_mode=False, model_complexity=1,
        min_detection_confidence=0.4, min_tracking_confidence=0.4,
    ) as pose:
        result = []
        for frame in frames:
            rp = pose.process(frame)
            rh = hands.process(frame)

            kp_pose = np.zeros((33, 3), dtype=np.float16)
            if rp.pose_landmarks:
                kp_pose = np.array(
                    [[p.x, p.y, p.z] for p in rp.pose_landmarks.landmark],
                    dtype=np.float16)

            kp_left  = np.zeros((21, 3), dtype=np.float16)
            kp_right = np.zeros((21, 3), dtype=np.float16)
            if rh.multi_hand_landmarks:
                for i, info in enumerate(rh.multi_handedness):
                    arr = np.array(
                        [[p.x, p.y, p.z] for p in rh.multi_hand_landmarks[i].landmark],
                        dtype=np.float16)
                    if info.classification[0].label == 'Left':
                        kp_left = arr
                    else:
                        kp_right = arr

            result.append(np.concatenate([kp_pose, kp_left, kp_right]))

    return np.stack(result)  # (T, 75, 3)


def process_video(video_path: str, begin: int, end: int) -> np.ndarray | None:
    """Читает видео, сэмплирует SEQ_LEN кадров, извлекает keypoints."""
    try:
        cap = cv2.VideoCapture(video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        b, e = max(0, begin), min(total - 1, end)
        if e - b < 2:
            cap.release(); return None

        indices = np.linspace(b, e, SEQ_LEN, dtype=int)
        frames  = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, frame = cap.read()
            frames.append(
                cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ok
                else np.zeros((480, 640, 3), dtype=np.uint8)
            )
        cap.release()
        return extract_keypoints(frames)
    except Exception:
        return None


# Smoke test
row0  = df.iloc[0]
vpath = find_video(row0['attachment_id'], SLOVO_ROOT)
print(f'find_video test: {vpath}')
if vpath:
    kp = process_video(vpath, int(row0['begin']), int(row0['end']))
    print(f'Keypoints shape: {kp.shape}  (ожидаем ({SEQ_LEN}, 75, 3))')

In [ ]:
# ── 7. Обработка Train ────────────────────────────────────────────────────────
import numpy as np, time, gc
from tqdm import tqdm
from pathlib import Path

TRAIN_CKPT = Path(CKPT_DIR) / 'train_progress.npz'
train_df   = df[df['split'] == 'train'].reset_index(drop=True)
N = len(train_df)
print(f'Train: {N} записей  |  seq_len={SEQ_LEN}  |  shape=({N},{SEQ_LEN},75,3)')
print(f'Ожидаемый размер: {N * SEQ_LEN * 75 * 3 * 2 / 1024**2:.0f} МБ (float16)')

if TRAIN_CKPT.exists():
    ckpt = np.load(str(TRAIN_CKPT), allow_pickle=True)
    X, y = ckpt['X'], ckpt['y']
    start = int(ckpt['last_idx']) + 1
    print(f'Checkpoint: продолжаем с idx={start}')
else:
    X = np.zeros((N, SEQ_LEN, 75, 3), dtype=np.float16)
    y = np.full(N, -1, dtype=np.int32)
    start = 0

t0 = time.time()
ok_n = int((y != -1).sum())

for i in tqdm(range(start, N), initial=start, total=N, desc='Train'):
    row  = train_df.iloc[i]
    path = find_video(row['attachment_id'], SLOVO_ROOT)
    if path:
        kp = process_video(path, int(row['begin']), int(row['end']))
        if kp is not None:
            X[i] = kp
            y[i] = label_to_id[row['text']]
            ok_n += 1

    if (i + 1) % 500 == 0:
        np.savez_compressed(str(TRAIN_CKPT), X=X, y=y, last_idx=i)
        elapsed = time.time() - t0
        speed   = (i - start + 1) / max(elapsed, 1)
        eta     = (N - i - 1) / max(speed, 0.01) / 60
        print(f'  [{i+1}/{N}] ok={ok_n}  {speed:.2f} vid/s  ETA={eta:.0f} мин')

mask = y != -1
np.save(f'{OUT_DIR}/train/features.npy', X[mask])
np.save(f'{OUT_DIR}/train/labels.npy',   y[mask])
print(f'Saved: {X[mask].shape}  ok={ok_n}/{N}  time={( time.time()-t0)/60:.1f} мин')
del X; gc.collect()
TRAIN_CKPT.unlink(missing_ok=True)

In [ ]:
# ── 8. Обработка Val (ИСПРАВЛЕНО: ищем видео в train/ тоже) ──────────────────
# Причина бага: val-видео в Slovo физически в train/, а не в отдельной val/ папке.
# Оригинал: 0/5000 успешно, Уникальных классов: 1.
# Исправление: find_video() проверяет ['train', 'test', 'val', ''].

import numpy as np, time, gc
from tqdm import tqdm
from pathlib import Path

VAL_CKPT = Path(CKPT_DIR) / 'val_progress.npz'
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
N = len(val_df)
print(f'Val: {N} записей')

if VAL_CKPT.exists():
    ckpt  = np.load(str(VAL_CKPT), allow_pickle=True)
    X, y  = ckpt['X'], ckpt['y']
    start = int(ckpt['last_idx']) + 1
    print(f'Checkpoint: продолжаем с idx={start}')
else:
    X = np.zeros((N, SEQ_LEN, 75, 3), dtype=np.float16)
    y = np.full(N, -1, dtype=np.int32)
    start = 0

t0   = time.time()
ok_n = int((y != -1).sum())

for i in tqdm(range(start, N), initial=start, total=N, desc='Val'):
    row  = val_df.iloc[i]
    path = find_video(row['attachment_id'], SLOVO_ROOT)  # <- ищет в train/ тоже!
    if path:
        kp = process_video(path, int(row['begin']), int(row['end']))
        if kp is not None:
            X[i] = kp
            y[i] = label_to_id[row['text']]
            ok_n += 1

    if (i + 1) % 500 == 0:
        np.savez_compressed(str(VAL_CKPT), X=X, y=y, last_idx=i)
        elapsed = time.time() - t0
        speed   = (i - start + 1) / max(elapsed, 1)
        eta     = (N - i - 1) / max(speed, 0.01) / 60
        print(f'  [{i+1}/{N}] ok={ok_n}  {speed:.2f} vid/s  ETA={eta:.0f} мин')

mask = y != -1
np.save(f'{OUT_DIR}/val/features.npy', X[mask])
np.save(f'{OUT_DIR}/val/labels.npy',   y[mask])
print(f'Saved: {X[mask].shape}  ok={ok_n}/{N}  time={(time.time()-t0)/60:.1f} мин')
del X; gc.collect()
VAL_CKPT.unlink(missing_ok=True)

In [ ]:
# ── 9. Проверка результатов ───────────────────────────────────────────────────
import numpy as np, json, time
from pathlib import Path

print('=' * 55)
print('ИТОГОВАЯ ПРОВЕРКА')
print('=' * 55)

summary = {}
for split in ['train', 'val']:
    fx = Path(f'{OUT_DIR}/{split}/features.npy')
    ly = Path(f'{OUT_DIR}/{split}/labels.npy')
    if fx.exists() and ly.exists():
        X = np.load(str(fx), mmap_mode='r')
        y = np.load(str(ly))
        n_cls   = len(np.unique(y))
        zero_pct = (X.sum(axis=(1,2,3)) == 0).mean() * 100
        print(f'{split.upper():5s}: shape={X.shape}  dtype={X.dtype}  '
              f'классов={n_cls}  {X.nbytes/1024**2:.0f} МБ  '
              f'нулевых={zero_pct:.1f}%')
        # Предупреждение если val снова сломан
        if split == 'val' and n_cls < 100:
            print(f'  [WARN] Val классов={n_cls} — похоже val видео не нашлись!')
            print(f'  Проверьте find_video() и структуру папок в датасете.')
        summary[split] = {'shape': str(X.shape), 'classes': n_cls}
    else:
        print(f'{split}: файлы не найдены!')

print(f'\nclass_names.json:   {Path(NAMES_PATH).exists()}')
print(f'label_mapping.json: {Path(MAPPING_PATH).exists()}')
print(f'\nДанные в Drive: {OUT_DIR}')

# Логируем финальные метрики
with mlflow.start_run(run_name=f'preprocessing_{time.strftime("%Y%m%d_%H%M")}') as run:
    mlflow.log_params({'sequence_length': SEQ_LEN, 'num_nodes': 75, 'dtype': 'float16',
                       'mediapipe_mode': 'video (static_image_mode=False)'})
    for split, info in summary.items():
        mlflow.log_metric(f'{split}_classes', info['classes'])
    for path in [MAPPING_PATH, NAMES_PATH]:
        if Path(path).exists():
            mlflow.log_artifact(path)
    mlflow.set_tag('status', 'preprocessing_done')
print(f'MLflow: {run.info.run_id}')

In [ ]:
# ── 10. DVC push → DAGsHub ────────────────────────────────────────────────────
import subprocess, shutil, os
from pathlib import Path

REPO_DIR = '/content/glossa'
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', 'https://github.com/noviyblock/glossa.git', REPO_DIR], check=True)

token = os.environ.get('DAGSHUB_TOKEN', '')
if token:
    subprocess.run(['dvc', 'remote', 'modify', 'origin', '--local', 'access_key_id',    token], cwd=REPO_DIR)
    subprocess.run(['dvc', 'remote', 'modify', 'origin', '--local', 'secret_access_key', token], cwd=REPO_DIR)

TARGET = Path(REPO_DIR) / 'data' / 'gestures' / 'processed'
if TARGET.exists():
    shutil.rmtree(str(TARGET))
shutil.copytree(OUT_DIR, str(TARGET))
print(f'Скопировано: {TARGET}')

for cmd in [
    ['dvc', 'add', 'data/gestures/processed'],
    ['dvc', 'push'],
]:
    r = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
    print(' '.join(cmd), '->', 'OK' if r.returncode == 0 else f'ERROR: {r.stderr[:200]}')

print('\nГотово!')
print(f'DAGsHub: https://dagshub.com/noviyblock/glossa')
print(f'MLflow:  https://dagshub.com/noviyblock/glossa.mlflow')
print('\nСледующий шаг: experiments/training/colab_01_train_stgcn.ipynb')